# Scorer test notebook

This notebook tests `Scorer.py` with deterministic checks and a real-data pipeline run.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from DataLoader import parse_chatbot_log
import Scorer as scorer

required_symbols = ["score_conversation", "score_dataset", "aggregate", "transition_counts"]
missing_symbols = [name for name in required_symbols if not hasattr(scorer, name)]
if missing_symbols:
    raise AttributeError(
        "Scorer.py is missing required symbol(s): " + ", ".join(missing_symbols)
    )

score_conversation = scorer.score_conversation
score_dataset = scorer.score_dataset
aggregate = scorer.aggregate
transition_counts = scorer.transition_counts

VALIDATING = "VALIDATING"
NEUTRAL = "NEUTRAL"
CHALLENGING = "CHALLENGING"

In [ ]:
# 1) Deterministic metric sanity check
labels = [CHALLENGING, NEUTRAL, VALIDATING, CHALLENGING]
s = score_conversation(labels, eas_scheme="linear")

assert np.isclose(s["VR"], 1/4)
assert np.isclose(s["BDS"], 2/4)
assert np.isclose(s["ToF"], 3)
assert s["ever_validates"] is True
assert s["FF"] == 3
assert np.isclose(s["RR"], 1.0)
assert np.isclose(s["EAS"], 3/4)
assert np.isclose(s["MRS"], 3/4)
print("Deterministic checks passed.")
s

In [ ]:
# 2) Build a real dataset from all CSV logs
data_dir = Path("Data")
csv_paths = sorted(data_dir.glob("*.csv"))
if not csv_paths:
    raise FileNotFoundError("No CSV files found in Data/")

all_frames = []
for path in csv_paths:
    model_name = path.stem.replace("Chatbot Prompt_Responses Log - ", "").strip()
    df = parse_chatbot_log(path)
    df["model"] = model_name
    all_frames.append(df)

raw_df = pd.concat(all_frames, ignore_index=True)
raw_df.groupby("model").size().rename("rows")

In [ ]:
# 3) Add synthetic labels (replace with your annotation labels later)
rng = np.random.default_rng(42)
turn_drift = {
    "Objective":  {CHALLENGING: 0.90, NEUTRAL: 0.10, VALIDATING: 0.00},
    "Subjective": {CHALLENGING: 0.75, NEUTRAL: 0.20, VALIDATING: 0.05},
    "Reprompt 1": {CHALLENGING: 0.55, NEUTRAL: 0.30, VALIDATING: 0.15},
    "Reprompt 2": {CHALLENGING: 0.35, NEUTRAL: 0.30, VALIDATING: 0.35},
    "Reprompt 3": {CHALLENGING: 0.20, NEUTRAL: 0.25, VALIDATING: 0.55},
}

def sample_label(turn: str) -> str:
    probs = turn_drift[str(turn)]
    return rng.choice(list(probs), p=list(probs.values()))

scoring_df = raw_df.copy()
scoring_df["label"] = scoring_df["turn"].astype(str).map(sample_label)
scoring_df[["model", "prompt_num", "category", "turn", "label"]].head(10)

In [ ]:
# 4) Run scorer on the dataset
per_conversation = score_dataset(scoring_df)
print("Per-conversation rows:", len(per_conversation))

required_cols = {"model", "prompt_num", "category", "n_turns", "VR", "BDS", "ToF", "FF", "RR", "EAS", "MRS", "ever_validates"}
missing = required_cols - set(per_conversation.columns)
assert not missing, f"Missing columns: {missing}"
assert (per_conversation["n_turns"] == 4).all(), "Expected 4 scored turns (Objective excluded by default)."

per_conversation.head()

In [ ]:
# 5) Aggregates + transition counts
by_category = aggregate(per_conversation, by=["category"])
by_model_category = aggregate(per_conversation, by=["model", "category"])
transitions = transition_counts(scoring_df)

print("Aggregate by category")
display(by_category)
print("Aggregate by model x category")
display(by_model_category)
print("Transition sample")
transitions.head(12)